In [1]:
!nvidia-smi

!nvidia-smi

!pip install -U peft==0.10.0
!pip install -U accelerate==0.30.1
!pip install -U transformers datasets bitsandbytes
!pip install -q trl

import os
os.makedirs("logs", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
print("✅ Setup complete!")

Fri Nov  7 08:04:03 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P0             27W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from transformers import BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id
model.config.use_cache = False

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [6]:
from datasets import load_dataset
import json

# training_data.json はアップロード済みを想定
dataset = load_dataset("json", data_files="training_data.json")

print("✅ Dataset loaded:", dataset)

✅ Dataset loaded: DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 100
    })
})


In [10]:
def preprocess_function(examples):
    inputs = examples["input"]
    outputs = examples["output"]
    text_pairs = [f"User: {inp}\nBot: {out}" for inp, out in zip(inputs, outputs)]

    model_inputs = tokenizer(
        text_pairs,
        max_length=512,
        truncation=True,
        padding="max_length",
    )

    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

processed_dataset = dataset["train"].map(preprocess_function, batched=True)
print("✅ Tokenization complete!")

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

✅ Tokenization complete!


In [11]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ LoRA configuration applied.")

trainable params: 3,407,872 || all params: 7,245,139,968 || trainable%: 0.04703666202518836
✅ LoRA configuration applied.


In [12]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="outputs",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=100,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset,
    tokenizer=tokenizer
)

trainer.train()
print("✅ Training finished.")

/tmp/ipython-input-1296116397.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
10,3.787500
20,0.116700
30,0.091500
40,0.075600
50,0.075300
60,0.060000
70,0.058500
80,0.050500
90,0.048800
100,0.044500


✅ Training finished.


In [13]:
OUTPUT_DIR = "mistral7b_finetuned"
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to {OUTPUT_DIR}")

✅ Model saved to mistral7b_finetuned


In [15]:
from transformers import pipeline

OUTPUT_DIR = "outputs/mistral-lora"

pipe = pipeline(
    "text-generation",
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    device_map="auto"
)

prompt = "User: What are the latest trends in artificial intelligence?\nBot:"
result = pipe(prompt, max_new_tokens=100, do_sample=True, temperature=0.7)

print("💬 Generated Response:")
print(result[0]["generated_text"])

OSError: outputs/mistral-lora is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`